# LangGraph： 状态图和持久执行

LangGraph 是2026年低等级状态编排的参考实现。Agent是一个状态机，节点是函数，边是转移。状态在每步之后是不可变而且检查点备份的。碰到失败时从上一个检查点恢复。

## 问题描述

Agents 和 工作流面临同一个问题： 当一个40步的执行在第38步出错，你希望重新回到第38步，而不是从头开始。

LangGraph的设计答案是：状态是一等类型对象，修改是显式的，每个节点后检查点都被持久化。恢复就是一次`load_state(session_id)`的调用。

## 基本概念

### 图

一张图由以下元素定义
- 状态类型。  一个类型字典（或者Pydantic 模型），所有节点都可以读取和修改。
- 节点。  函数，从`state`生产`state_update`，之后状态更新被增量合并进原状态。
- 边。  节点间的条件或直接转移。
- 进入和退出。   `START` 和 `END` 哨兵节点用来标记边界。

### 持久执行

在每个节点执行返回后，运行时将状态序列化然后写入检查点。低N步失败了，运行时可以`resume(session_id)`，然后从第N+1步带着精确的状态跑。

重要的不是图的形状，而是图的形状加上检查点之后，恢复就变得廉价。

### 流式

每个节点可以吐出部分输出。图将每节点的增量事件流式吐出给调用者，所以UIs可以随图运行更新。

### 人在循环中

节点间检查和修改状态。实现方式是：在关键节点前暂定，将状态暴露给人，是否接受修改、还原。检查点让这件事变得很容易，因为状态已经序列化了。

### 记忆

短期（一轮————状态中的会话历史）和长期（跨轮次————通过检查点加上分离的长期存储做持久化）。LangGraph 通过工具集成外部记忆系统（Mem0， custom）等。

### 三种拓扑

1. 监督者。  中心路由LLM分发到具体的子agents。
2. Swarm/端到端。 Agents通过共享的工具接口直接传递信息。没有中心路由。
3. 层级。  监督者下辖子监督者，就像嵌套的子图一样。

### 什么时候这种模式出错

1. 检查点太小。  只对会话轮次做检查点会让工具状态和记忆写入无法恢复，完整状态必须序列话。
2. 非确定性节点。  恢复假设节点输入产生相同的状态更新。随机种子、始终、外部API都应该被捕获。
3. 过度使用条件边。 一张每条边都是条件边的图，是一个无法被推理的状态机。优先使用线性链加偶尔的分支。

# 开始编码

对应本章核心：**状态一等（TypedDict + 增量合并）**、**节点后检查点 / resume**、**流式事件**、**人在循环中（HITL）**、**监督者拓扑**。  
先用玩具状态机跑通检查点恢复与 HITL；再用 **PyTorch** 学监督者路由；最后用 **LangGraph + LangChain/DeepSeek** 跑可恢复生产图。


## 1. 教学玩具：状态图 + 检查点运行时

- **State**：`dict` 一等对象；节点只返回 partial update，由 reducer 合并。
- **Checkpoint**：每节点后序列化整状态（含工具/记忆字段）。
- **Resume**：失败或 HITL 暂停后从下一节点继续。
- **Stream**：每步吐出 `{node, update}`。
- **Supervisor**：中心路由到 research / draft worker。


In [1]:
from __future__ import annotations

import copy
import json
from dataclasses import dataclass, field
from typing import Any, Callable, Literal

NodeFn = Callable[[dict[str, Any]], dict[str, Any]]
EdgeFn = Callable[[dict[str, Any]], str]

START = "__start__"
END = "__end__"


def merge_state(state: dict[str, Any], update: dict[str, Any]) -> dict[str, Any]:
    """
    Args:
        state: 当前完整状态。
        update: 节点 partial update。

    Returns:
        next_state: 合并后的新状态（不可变风格：返回副本）。
    """
    out = copy.deepcopy(state)
    for k, v in update.items():
        if k == "log" and isinstance(v, list):
            out.setdefault("log", [])
            out["log"] = list(out["log"]) + list(v)
        else:
            out[k] = copy.deepcopy(v)
    return out


@dataclass
class Checkpoint:
    """一次节点执行后的检查点。"""

    step: int
    node: str
    next_node: str
    state: dict[str, Any]


@dataclass
class GraphRun:
    """一次图执行的结果。"""

    status: Literal["ok", "failed", "interrupted"]
    state: dict[str, Any]
    events: list[dict[str, Any]] = field(default_factory=list)
    error: str | None = None
    session_id: str = ""


class ToyStateGraph:
    """微型 LangGraph：节点 / 边 / 检查点 / HITL / 流式。"""

    def __init__(self) -> None:
        self.nodes: dict[str, NodeFn] = {}
        self.edges: dict[str, str | EdgeFn] = {}
        self.entry: str = END

    def add_node(self, name: str, fn: NodeFn) -> None:
        """
        Args:
            name: 节点名。
            fn: ``state -> partial_update``。
        """
        self.nodes[name] = fn

    def add_edge(self, src: str, dst: str) -> None:
        """固定边。"""
        self.edges[src] = dst

    def add_conditional_edges(self, src: str, router: EdgeFn) -> None:
        """条件边。"""
        self.edges[src] = router

    def set_entry(self, name: str) -> None:
        self.entry = name

    def _next(self, src: str, state: dict[str, Any]) -> str:
        edge = self.edges.get(src, END)
        if callable(edge):
            return edge(state)
        return edge


class CheckpointRuntime:
    """每个节点后持久化完整状态；支持 fail / HITL / resume。"""

    def __init__(self) -> None:
        self.store: dict[str, list[Checkpoint]] = {}

    def save(self, session_id: str, cp: Checkpoint) -> None:
        self.store.setdefault(session_id, []).append(cp)

    def load(self, session_id: str) -> Checkpoint | None:
        """
        Args:
            session_id: 会话 ID。

        Returns:
            cp: 最新检查点或 None。
        """
        cps = self.store.get(session_id) or []
        return cps[-1] if cps else None

    def run(
        self,
        graph: ToyStateGraph,
        state: dict[str, Any],
        session_id: str,
        *,
        fail_at: str | None = None,
        stream: bool = True,
    ) -> GraphRun:
        """
        Args:
            graph: 玩具图。
            state: 初始状态（完整，含 tool_state / memory）。
            session_id: 检查点会话。
            fail_at: 模拟在该节点抛错（测 resume）。
            stream: 是否记录流式事件。

        Returns:
            result: ok / failed / interrupted。
        """
        events: list[dict[str, Any]] = []
        cur = copy.deepcopy(state)
        node = graph.entry
        step = 0
        while node != END:
            if node not in graph.nodes:
                return GraphRun("failed", cur, events, f"unknown node {node}", session_id)
            if fail_at == node:
                return GraphRun("failed", cur, events, f"boom at {node}", session_id)
            try:
                update = graph.nodes[node](cur)
            except InterruptedError as e:
                # HITL：保存待审批状态
                nxt = graph._next(node, cur)
                self.save(session_id, Checkpoint(step, node, nxt, copy.deepcopy(cur)))
                payload = e.args[0] if e.args else {}
                if stream:
                    events.append({"node": node, "interrupt": payload})
                return GraphRun("interrupted", cur, events, None, session_id)
            cur = merge_state(cur, update)
            nxt = graph._next(node, cur)
            self.save(session_id, Checkpoint(step, node, nxt, copy.deepcopy(cur)))
            if stream:
                events.append({"node": node, "update": update, "next": nxt})
            node = nxt
            step += 1
            if step > 32:
                return GraphRun("failed", cur, events, "max steps", session_id)
        return GraphRun("ok", cur, events, None, session_id)

    def resume(
        self,
        graph: ToyStateGraph,
        session_id: str,
        *,
        patch: dict[str, Any] | None = None,
        skip_current: bool = False,
        fail_at: str | None = None,
    ) -> GraphRun:
        """
        Args:
            graph: 玩具图。
            session_id: 会话。
            patch: HITL 对人状态的修改（approve / rewrite）。
            skip_current: True 则从检查点的 next_node 继续（节点已成功）。
            fail_at: 恢复路径上再模拟失败。

        Returns:
            result: 继续执行结果。
        """
        cp = self.load(session_id)
        if cp is None:
            return GraphRun("failed", {}, [], "no checkpoint", session_id)
        state = merge_state(cp.state, patch or {})
        start = cp.next_node if skip_current else cp.node
        # 临时改 entry 从恢复点跑
        old_entry = graph.entry
        graph.entry = start
        try:
            return self.run(graph, state, session_id, fail_at=fail_at)
        finally:
            graph.entry = old_entry


def build_supervisor_graph() -> ToyStateGraph:
    """
    Returns:
        graph: 监督者 → worker → finalize。
    """
    g = ToyStateGraph()

    def supervisor(s: dict[str, Any]) -> dict[str, Any]:
        task = s.get("task", "")
        route = "research" if "research" in task.lower() or "查" in task else "draft"
        return {"route": route, "log": [f"supervisor:{route}"], "tool_state": {"routed": route}}

    def research(s: dict[str, Any]) -> dict[str, Any]:
        notes = ["fact:A", "fact:B"]
        mem = dict(s.get("memory") or {})
        mem["facts"] = notes
        return {"notes": notes, "memory": mem, "log": ["research"]}

    def draft(s: dict[str, Any]) -> dict[str, Any]:
        facts = (s.get("memory") or {}).get("facts") or s.get("notes") or ["(no facts)"]
        text = f"DRAFT[{s.get('route')}]: " + "; ".join(facts)
        return {"draft": text, "log": ["draft"]}

    def hitl(s: dict[str, Any]) -> dict[str, Any]:
        if not s.get("approved"):
            raise InterruptedError({"ask": "approve draft?", "draft": s.get("draft")})
        return {"log": ["hitl:pass"]}

    def finalize(s: dict[str, Any]) -> dict[str, Any]:
        return {"final": s.get("draft", "") + " | FINAL", "log": ["finalize"]}

    g.add_node("supervisor", supervisor)
    g.add_node("research", research)
    g.add_node("draft", draft)
    g.add_node("hitl", hitl)
    g.add_node("finalize", finalize)
    g.set_entry("supervisor")

    def after_supervisor(s: dict[str, Any]) -> str:
        return "research" if s.get("route") == "research" else "draft"

    g.add_conditional_edges("supervisor", after_supervisor)
    g.add_edge("research", "draft")
    g.add_edge("draft", "hitl")
    g.add_edge("hitl", "finalize")
    g.add_edge("finalize", END)
    return g


print("ToyStateGraph ready | checkpoint + HITL + supervisor")


ToyStateGraph ready | checkpoint + HITL + supervisor


## 2. 玩具示例：失败恢复、HITL、完整检查点


In [2]:
def demo_langgraph_toy() -> None:
    """断言：流式、失败 resume、HITL、完整状态检查点。"""
    g = build_supervisor_graph()
    rt = CheckpointRuntime()
    init = {
        "task": "research LangGraph checkpoints",
        "approved": False,
        "log": [],
        "memory": {},
        "tool_state": {},
    }

    # 1) 跑到 draft 前失败 → 检查点含 memory；resume 从 draft 进 HITL
    r1 = rt.run(g, init, "s-fail", fail_at="draft")
    assert r1.status == "failed" and r1.error and "draft" in r1.error
    cp = rt.load("s-fail")
    assert cp is not None and cp.node == "research" and cp.next_node == "draft"
    assert "facts" in (cp.state.get("memory") or {})
    print("fail@draft:", r1.error, "cp=", cp.node, "->", cp.next_node)

    r2 = rt.resume(g, "s-fail", skip_current=True)
    assert r2.status == "interrupted"
    cp2 = rt.load("s-fail")
    assert cp2 is not None and cp2.node == "hitl"
    print("resume→HITL:", r2.events[-1])

    # 2) 人批准并改写 draft，重跑 hitl → finalize
    r3 = rt.resume(
        g,
        "s-fail",
        patch={"approved": True, "draft": "HUMAN_EDITED_DRAFT"},
        skip_current=False,
    )
    assert r3.status == "ok"
    assert r3.state.get("final", "").startswith("HUMAN_EDITED_DRAFT")
    print("HITL approved final:", r3.state["final"])

    # 3) 干净跑通 + 流式事件
    rt2 = CheckpointRuntime()
    clean = rt2.run(
        g,
        {**init, "task": "write a short note", "approved": True},
        "s-ok",
    )
    assert clean.status == "ok"
    nodes = [e["node"] for e in clean.events if "update" in e]
    assert nodes == ["supervisor", "draft", "hitl", "finalize"]
    print("stream nodes:", nodes)

    # 4) 检查点太小的反例
    tiny = {"messages": ["hi"]}
    assert "tool_state" not in tiny and "memory" not in tiny
    print("tiny-checkpoint anti-pattern: missing tool_state/memory")
    print("TOY DEMO OK")


demo_langgraph_toy()


fail@draft: boom at draft cp= research -> draft
resume→HITL: {'node': 'hitl', 'interrupt': {'ask': 'approve draft?', 'draft': 'DRAFT[research]: fact:A; fact:B'}}
HITL approved final: HUMAN_EDITED_DRAFT | FINAL
stream nodes: ['supervisor', 'draft', 'hitl', 'finalize']
tiny-checkpoint anti-pattern: missing tool_state/memory
TOY DEMO OK


## 3. PyTorch：监督者路由头

监督者拓扑的核心是「中心路由 → worker」。用小型 MLP 学 research / draft / code 三路分发。


In [3]:
import re

import torch
import torch.nn as nn
import torch.nn.functional as F

SUP_LABELS = ["research", "draft", "code"]
SUP_VOCAB = [
    "research",
    "search",
    "fact",
    "draft",
    "write",
    "summary",
    "code",
    "bug",
    "python",
    "fix",
]


def sup_featurize(text: str) -> torch.Tensor:
    """
    Args:
        text: 监督者看到的任务。

    Returns:
        x: ``(V,)`` 多热。
    """
    toks = set(re.findall(r"[a-z]+", text.lower()))
    x = torch.zeros(len(SUP_VOCAB))
    for i, w in enumerate(SUP_VOCAB):
        if w in toks:
            x[i] = 1.0
    return x


class SupervisorNet(nn.Module):
    """监督者路由小网络。"""

    def __init__(self) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(len(SUP_VOCAB), 16),
            nn.ReLU(),
            nn.Linear(16, len(SUP_LABELS)),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def train_supervisor(steps: int = 250) -> SupervisorNet:
    """
    Args:
        steps: 训练步数。

    Returns:
        model: 路由头。
    """
    samples = [
        ("research search fact about checkpoint", 0),
        ("search fact database", 0),
        ("write draft summary", 1),
        ("draft a short write summary", 1),
        ("fix python bug in code", 2),
        ("python code fix bug", 2),
    ]
    X = torch.stack([sup_featurize(t) for t, _ in samples])
    y = torch.tensor([i for _, i in samples])
    model = SupervisorNet()
    opt = torch.optim.Adam(model.parameters(), lr=0.05)
    for _ in range(steps):
        loss = F.cross_entropy(model(X), y)
        opt.zero_grad()
        loss.backward()
        opt.step()
    return model


def predict_worker(model: SupervisorNet, text: str) -> str:
    """
    Args:
        model: 监督者网络。
        text: 任务。

    Returns:
        worker: research / draft / code。
    """
    with torch.no_grad():
        idx = int(model(sup_featurize(text)).argmax().item())
    return SUP_LABELS[idx]


def demo_pytorch_supervisor() -> None:
    model = train_supervisor()
    queries = [
        "research search fact persistence",
        "write draft summary for blog",
        "fix python bug in code",
    ]
    print("=== supervisor router ===")
    for q in queries:
        print(f"{predict_worker(model, q):>8}  <-  {q}")
    assert predict_worker(model, "fix python bug in code") == "code"
    print("PYTORCH DEMO OK")


demo_pytorch_supervisor()


=== supervisor router ===
research  <-  research search fact persistence
   draft  <-  write draft summary for blog
    code  <-  fix python bug in code
PYTORCH DEMO OK


## 4. 生产级：LangGraph 可恢复图 + DeepSeek

真实 ``StateGraph`` + ``MemorySaver``：supervisor → worker → HITL(`interrupt`) → finalize。  
再用 **LangChain `create_agent`** 暴露 `run_graph` / `approve_resume` / `get_checkpoint` 工具。需 ``DEEPSEEK_API_KEY``。


In [4]:
import json
import operator
import os
import sys
from pathlib import Path
from typing import Annotated, Any, Literal, TypedDict

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import StructuredTool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt
from pydantic import BaseModel, Field

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import load_project_env  # noqa: E402

load_project_env()

MODEL = "deepseek:deepseek-v4-flash"
SUP_MODEL = train_supervisor(steps=120)


class AgentState(TypedDict):
    """图状态：一等 TypedDict；log 用 append reducer。"""

    task: str
    route: str
    notes: str
    draft: str
    final: str
    approved: bool
    memory: dict[str, Any]
    tool_state: dict[str, Any]
    log: Annotated[list[str], operator.add]


def get_llm(*, temperature: float = 0.0) -> Any:
    """
    Returns:
        llm: DeepSeek chat model。
    """
    if not os.getenv("DEEPSEEK_API_KEY"):
        raise RuntimeError("DEEPSEEK_API_KEY missing; copy .env.example → .env")
    return init_chat_model(
        MODEL,
        temperature=temperature,
        extra_body={"thinking": {"type": "disabled"}},
    )


def node_supervisor(state: AgentState) -> dict[str, Any]:
    """监督者：PyTorch 路由 + 记录 tool_state。"""
    route = predict_worker(SUP_MODEL, state["task"])
    return {
        "route": route,
        "tool_state": {**(state.get("tool_state") or {}), "routed": route},
        "log": [f"supervisor:{route}"],
    }


def node_research(state: AgentState) -> dict[str, Any]:
    """research worker：LLM 抽要点。"""
    prompt = (
        f"Task: {state['task']}\n"
        "List 2 short factual bullets in Chinese, one line each. No preamble."
    )
    notes = str(get_llm().invoke(prompt).content).strip()
    mem = dict(state.get("memory") or {})
    mem["facts"] = notes
    return {"notes": notes, "memory": mem, "log": ["research"]}


def node_draft(state: AgentState) -> dict[str, Any]:
    """draft worker：基于 notes/memory 写短稿。"""
    facts = state.get("notes") or (state.get("memory") or {}).get("facts") or ""
    prompt = (
        f"Task: {state['task']}\nNotes:\n{facts}\n"
        "Write a <=80 Chinese char draft. No markdown headings."
    )
    draft = str(get_llm().invoke(prompt).content).strip()
    return {"draft": draft, "log": ["draft"]}


def node_code(state: AgentState) -> dict[str, Any]:
    """code worker：给伪代码要点。"""
    prompt = f"Task: {state['task']}\nGive 3 bullet pseudo-steps in Chinese."
    notes = str(get_llm().invoke(prompt).content).strip()
    return {"notes": notes, "draft": f"CODE_PLAN: {notes[:120]}", "log": ["code"]}


def node_hitl(state: AgentState) -> dict[str, Any]:
    """人在循环：interrupt 直到 resume 带 ok。"""
    if state.get("approved"):
        return {"log": ["hitl:already_approved"]}
    decision = interrupt({"ask": "approve draft?", "draft": state.get("draft", "")})
    ok = bool(decision.get("ok")) if isinstance(decision, dict) else bool(decision)
    patch: dict[str, Any] = {"approved": ok, "log": ["hitl:resume"]}
    if isinstance(decision, dict) and decision.get("draft"):
        patch["draft"] = decision["draft"]
    return patch


def node_finalize(state: AgentState) -> dict[str, Any]:
    """定稿。"""
    return {"final": (state.get("draft") or "") + " | FINAL", "log": ["finalize"]}


def route_after_supervisor(state: AgentState) -> Literal["research", "draft", "code"]:
    r = state.get("route") or "draft"
    if r in ("research", "draft", "code"):
        return r  # type: ignore[return-value]
    return "draft"


def build_langgraph_app():
    """
    Returns:
        app: 带 MemorySaver 的编译图。
    """
    g = StateGraph(AgentState)
    g.add_node("supervisor", node_supervisor)
    g.add_node("research", node_research)
    g.add_node("draft", node_draft)
    g.add_node("code", node_code)
    g.add_node("hitl", node_hitl)
    g.add_node("finalize", node_finalize)
    g.add_edge(START, "supervisor")
    g.add_conditional_edges(
        "supervisor",
        route_after_supervisor,
        {"research": "research", "draft": "draft", "code": "code"},
    )
    g.add_edge("research", "draft")
    g.add_edge("draft", "hitl")
    g.add_edge("code", "hitl")
    g.add_edge("hitl", "finalize")
    g.add_edge("finalize", END)
    return g.compile(checkpointer=MemorySaver())


PROD_APP = build_langgraph_app()
PROD_THREAD = "lg-demo-1"


def reset_prod_graph() -> None:
    """新 thread + 新 checkpointer。"""
    global PROD_APP, PROD_THREAD
    PROD_APP = build_langgraph_app()
    PROD_THREAD = f"lg-demo-{os.urandom(3).hex()}"


class RunGraphArgs(BaseModel):
    task: str = Field(description="User task for the state graph")


class ApproveArgs(BaseModel):
    ok: bool = Field(description="Whether human approves the draft")
    draft: str | None = Field(default=None, description="Optional rewritten draft")


class GetStateArgs(BaseModel):
    pass


def _cfg() -> dict[str, Any]:
    return {"configurable": {"thread_id": PROD_THREAD}}


def run_graph_impl(task: str) -> str:
    """
    启动图直到 END 或 HITL interrupt。

    Returns:
        json: status + state snapshot。
    """
    init: AgentState = {
        "task": task,
        "route": "",
        "notes": "",
        "draft": "",
        "final": "",
        "approved": False,
        "memory": {},
        "tool_state": {},
        "log": [],
    }
    result = PROD_APP.invoke(init, _cfg())
    interrupted = bool(result.get("__interrupt__"))
    payload = {
        "status": "interrupted" if interrupted else "ok",
        "thread_id": PROD_THREAD,
        "route": result.get("route"),
        "draft": result.get("draft"),
        "final": result.get("final"),
        "memory": result.get("memory"),
        "tool_state": result.get("tool_state"),
        "log": result.get("log"),
        "interrupt": [str(x) for x in (result.get("__interrupt__") or [])],
    }
    return json.dumps(payload, ensure_ascii=False)


def approve_resume_impl(ok: bool, draft: str | None = None) -> str:
    """
    HITL 批准后从检查点 resume。

    Returns:
        json: 恢复后状态。
    """
    payload: dict[str, Any] = {"ok": ok}
    if draft:
        payload["draft"] = draft
    result = PROD_APP.invoke(Command(resume=payload), _cfg())
    return json.dumps(
        {
            "status": "ok",
            "final": result.get("final"),
            "draft": result.get("draft"),
            "approved": result.get("approved"),
            "log": result.get("log"),
            "memory": result.get("memory"),
        },
        ensure_ascii=False,
    )


def get_checkpoint_impl() -> str:
    """
    Returns:
        json: 当前 thread 状态快照。
    """
    snap = PROD_APP.get_state(_cfg())
    values = snap.values if snap else {}
    return json.dumps(
        {
            "thread_id": PROD_THREAD,
            "next": list(snap.next) if snap else [],
            "values": {
                k: values.get(k)
                for k in ["task", "route", "draft", "final", "approved", "memory", "tool_state", "log"]
            },
        },
        ensure_ascii=False,
    )


def build_graph_tools() -> list[StructuredTool]:
    """
    Returns:
        tools: 图控制三件套。
    """

    def _run(**kwargs: Any) -> str:
        return run_graph_impl(RunGraphArgs(**kwargs).task)

    def _approve(**kwargs: Any) -> str:
        a = ApproveArgs(**kwargs)
        return approve_resume_impl(a.ok, a.draft)

    def _state(**kwargs: Any) -> str:
        return get_checkpoint_impl()

    return [
        StructuredTool.from_function(
            name="run_graph",
            description="Start LangGraph until END or HITL interrupt. Persists checkpoint.",
            func=_run,
            args_schema=RunGraphArgs,
        ),
        StructuredTool.from_function(
            name="approve_resume",
            description="Human-in-the-loop: approve/reject and resume from checkpoint.",
            func=_approve,
            args_schema=ApproveArgs,
        ),
        StructuredTool.from_function(
            name="get_checkpoint",
            description="Load current thread checkpoint (full state including memory/tool_state).",
            func=_state,
            args_schema=GetStateArgs,
        ),
    ]


GRAPH_TOOLS = build_graph_tools()


def build_control_agent() -> Any:
    """
    Returns:
        agent: 通过工具驱动 LangGraph。
    """
    system = (
        "You control a LangGraph state machine via tools.\n"
        "Flow: run_graph(task) -> if interrupted, get_checkpoint then approve_resume(ok=true).\n"
        "Do not invent state; always use tools. Reply in Chinese when done."
    )
    return create_agent(get_llm(), GRAPH_TOOLS, system_prompt=system)


def format_agent_messages(messages: list[BaseMessage]) -> str:
    lines: list[str] = []
    for m in messages:
        if isinstance(m, HumanMessage):
            lines.append(f"USER: {m.content}")
        elif isinstance(m, AIMessage):
            if m.tool_calls:
                for tc in m.tool_calls:
                    lines.append(f"ACTION: {tc['name']}({tc.get('args') or {}})")
            if m.content:
                lines.append(f"ASSISTANT: {m.content}")
        elif isinstance(m, ToolMessage):
            content = m.content if len(str(m.content)) < 500 else str(m.content)[:500] + "..."
            lines.append(f"OBS[{m.name}]: {content}")
    return "\n".join(lines)


def count_tool_calls(messages: list[BaseMessage]) -> int:
    n = 0
    for m in messages:
        if isinstance(m, AIMessage) and m.tool_calls:
            n += len(m.tool_calls)
    return n


print(f"LangGraph + LangChain ready | {MODEL}")


LangGraph + LangChain ready | deepseek:deepseek-v4-flash


## 5. 生产示例：run → HITL interrupt → approve_resume


In [5]:
def demo_deepseek_langgraph() -> None:
    """真实 API；无 key 则 SKIP。"""
    if not os.getenv("DEEPSEEK_API_KEY"):
        print("SKIP production demo: DEEPSEEK_API_KEY missing")
        return

    reset_prod_graph()
    # 直接测图：interrupt → resume（不依赖 agent 是否调用对工具）
    out1 = json.loads(
        run_graph_impl("research search fact: why checkpoint full state in LangGraph")
    )
    print("=== graph run ===")
    print(json.dumps(out1, ensure_ascii=False, indent=2)[:800])
    assert out1["status"] == "interrupted"
    assert out1.get("memory") or out1.get("draft")

    out2 = json.loads(approve_resume_impl(True))
    print("=== resume ===")
    print(json.dumps(out2, ensure_ascii=False, indent=2)[:600])
    assert out2.get("final") and "FINAL" in out2["final"]

    # agent 控制面：再跑一轮 draft 任务
    reset_prod_graph()
    agent = build_control_agent()
    result = agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content=(
                        "请用工具跑 LangGraph：任务是 write draft summary about HITL。"
                        "若 interrupt 则 approve_resume(ok=true)，最后用中文一句话总结 final。"
                    )
                )
            ]
        }
    )
    print("=== agent control ===")
    print(format_agent_messages(result["messages"]))
    assert count_tool_calls(result["messages"]) >= 2
    print("PRODUCTION DEMO OK")


demo_deepseek_langgraph()


=== graph run ===
{
  "status": "interrupted",
  "thread_id": "lg-demo-da483b",
  "route": "research",
  "draft": "LangGraph完整保存节点状态与输入输出，确保中断后精确恢复，避免重算；同时支持时间旅行调试与人工修正，提升复杂工作流的可观测性和可控性。",
  "final": "",
  "memory": {
    "facts": "- LangGraph的checkpoint完整保存节点状态、输入输出及配置，确保图执行中断后能从精确断点恢复，避免重算或状态丢失。  \n- 完整状态快照支持时间旅行调试和人工介入修正，便于回滚到任意历史步骤，提升复杂工作流的可观测性与可控性。"
  },
  "tool_state": {
    "routed": "research"
  },
  "log": [
    "supervisor:research",
    "research",
    "draft"
  ],
  "interrupt": [
    "Interrupt(value={'ask': 'approve draft?', 'draft': 'LangGraph完整保存节点状态与输入输出，确保中断后精确恢复，避免重算；同时支持时间旅行调试与人工修正，提升复杂工作流的可观测性和可控性。'}, id='9e5ad8c9d723ae57469054d9e80994a3')"
  ]
}
=== resume ===
{
  "status": "ok",
  "final": "LangGraph完整保存节点状态与输入输出，确保中断后精确恢复，避免重算；同时支持时间旅行调试与人工修正，提升复杂工作流的可观测性和可控性。 | FINAL",
  "draft": "LangGraph完整保存节点状态与输入输出，确保中断后精确恢复，避免重算；同时支持时间旅行调试与人工修正，提升复杂工作流的可观测性和可控性。",
  "approved": true,
  "log": [
    "supervisor:research",
    "research",
    "draft",
    "hitl:resume",
   